# Flight Data Analysis Notebook

This notebook processes and analyzes flight data for the year 2024. It builds core dimension tables for dates and airports, generates daily and weekly flight metrics, and produces performance summaries for airports based on flight volume, delays, and cancellations.

The cell below calculates the date dimension for the year 2024. The dimension includes the date key, year, month, day, day of the week, and whether the day is a weekend.

In [0]:
%sql
CREATE OR REPLACE TABLE task11.core.dim_date AS
WITH date_seq AS (
  SELECT
    sequence(
      to_date('2024-01-01'),
      to_date('2024-12-31'),
      interval 1 day
    ) AS all_dates
)
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key,
  d AS date,
  year(d) AS year,
  month(d) AS month,
  day(d) AS day,
  dayofweek(d) AS day_of_week,
  CASE WHEN dayofweek(d) IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekend
FROM date_seq
LATERAL VIEW explode(all_dates) AS d
ORDER BY date_key;



num_affected_rows,num_inserted_rows


The cell below calculates the airport dimension where the surrogate key is created and will be attached to the fact table. The dimension contains the airport code, the city and the state.

In [0]:
%sql
CREATE OR REPLACE TABLE task11.core.dim_airport AS
WITH airports AS (
  SELECT
      origin AS airport_code,
      MIN(origin_city_name) AS city,
      MIN(origin_state_nm) AS state
  FROM task11.core.silver_flights
  GROUP BY origin
)
SELECT
    ROW_NUMBER() OVER (ORDER BY airport_code) AS airport_key,
    airport_code,
    city,
    state
FROM airports;


num_affected_rows,num_inserted_rows


The cell below calculates the fact table for the flights daily from an airport. It also calculates the average time, distance the flight fly, cancellations, delayed flights, and the average delay flights might incur.

In [0]:
%sql
CREATE OR REPLACE TABLE task11.core.fact_flights_daily AS
SELECT
    d.date_key,
    a.airport_key,

    COUNT(*) AS total_flights,
    AVG(sf.air_time) AS avg_air_time,
    AVG(sf.distance) AS avg_distance,
    SUM(CAST(sf.cancelled AS INT)) AS cancellations,
    SUM(CASE WHEN sf.total_delay > 0 THEN 1 ELSE 0 END) AS delayed_flights,
    AVG(sf.total_delay) AS avg_total_delay

FROM task11.core.silver_flights sf

LEFT JOIN task11.core.dim_airport a
    ON sf.origin = a.airport_code  

LEFT JOIN task11.core.dim_date d
    ON sf.flight_date = d.date

GROUP BY
    d.date_key,
    a.airport_key

ORDER BY
    d.date_key;


num_affected_rows,num_inserted_rows


The cell below calculates the fact table for the flights by joining the date and airport dimensions to the table. The table shows if the flight was cancelled, delayed and the air time and distance it flew.

In [0]:
%sql
CREATE OR REPLACE TABLE task11.core.fact_flights AS
WITH flights_with_id AS (
    SELECT
        ROW_NUMBER() OVER (ORDER BY flight_date, origin) AS flight_id,  
        flight_date,
        origin,
        CAST(cancelled AS INT) AS cancelled,
        total_delay AS delay,
        air_time,
        distance
    FROM task11.core.silver_flights
)
SELECT
    f.flight_id,
    d.date_key AS flight_date_key,               
    a_origin.airport_key AS origin_airport_key,  
    f.cancelled,
    f.delay,
    f.air_time,
    f.distance
FROM flights_with_id f

LEFT JOIN task11.core.dim_airport a_origin
    ON f.origin = a_origin.airport_code

LEFT JOIN task11.core.dim_date d
    ON f.flight_date = d.date

ORDER BY
    d.date_key;

num_affected_rows,num_inserted_rows


The cell below makes a fact table with the flight metrics for an airport for one week. The dataset as of now has 9 weeks worth of data so one airport has 9 rows with the details for delays, cancellations, peak departure hour.

In [0]:
%sql
CREATE OR REPLACE TABLE task11.core.gold_airport_metrics_weekly AS
WITH weekly_peak_hour AS (
    SELECT
        origin,
        week_of_year,
        dep_hour AS peak_departure_hour
    FROM (
        SELECT
            origin,
            weekofyear(flight_date) AS week_of_year,
            dep_hour,
            COUNT(*) AS cnt,
            ROW_NUMBER() OVER (
                PARTITION BY origin, weekofyear(flight_date) 
                ORDER BY COUNT(*) DESC
            ) AS rn
        FROM task11.core.silver_flights
        GROUP BY origin, weekofyear(flight_date), dep_hour
    ) t
    WHERE rn = 1
)
SELECT
    a.airport_key,
    a.airport_code,
    a.city,
    a.state,
    weekofyear(f.flight_date) AS week_of_year,
    COUNT(*) AS total_flights,
    SUM(CASE WHEN is_delayed THEN 1 ELSE 0 END) AS delayed_flights,
    AVG(total_delay) AS avg_delay,
    SUM(CASE WHEN cancelled THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS cancellation_rate,
    p.peak_departure_hour
FROM task11.core.silver_flights f
LEFT JOIN task11.core.dim_airport a
    ON f.origin = a.airport_code
LEFT JOIN weekly_peak_hour p
    ON f.origin = p.origin AND weekofyear(f.flight_date) = p.week_of_year
GROUP BY a.airport_key, a.airport_code, a.city, a.state, weekofyear(f.flight_date), p.peak_departure_hour
ORDER BY a.airport_key, week_of_year;


num_affected_rows,num_inserted_rows


The cell below calculates the airport metrics for the best and worst performing airports. The calculation is divided into two parts, high and low volume airports. High-volume airports are those with more than 10,000 flights, and low-volume airports have fewer than 10,000 flights. The airport rank is calculated by the average delay and cancellation rate of the flights.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW fact_base AS
SELECT
    d.airport_key,
    d.airport_code,
    m.total_flights,
    m.delayed_flights,
    m.avg_delay,
    m.cancellation_rate,
    m.peak_departure_hour,
    RANK() OVER (ORDER BY m.cancellation_rate ASC) AS rank_by_cancellation,
    RANK() OVER (ORDER BY m.avg_delay ASC) AS rank_by_avg_delay
FROM task11.core.origin_metrics m
LEFT JOIN task11.core.dim_airport d
    ON m.origin = d.airport_code;

CREATE OR REPLACE TABLE task11.core.gold_airport_perf_high_volume AS
SELECT *
FROM fact_base
WHERE total_flights > 10000;

CREATE OR REPLACE TABLE task11.core.gold_airport_perf_low_volume AS
SELECT *
FROM fact_base
WHERE total_flights <= 10000;

